In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax
from torch_geometric.nn import HeteroConv, GATConv, HGTConv
from torch_geometric.nn import GCNConv, GATConv, GINConv
from torch_scatter import scatter

import pickle
import pandas as pd
import numpy as np

from helper import networkx_to_hetero_data, get_device, set_random_seeds
from typing import Any, Dict, Tuple
from typing import Any, Dict, List

#### load data

In [3]:
patient_kg_path = "./data/KG/patient_kg.pkl"

with open(patient_kg_path, 'rb') as f:
    G = pickle.load(f) # patient_kg

# prepare HeteroData
data, new_node_mappings = networkx_to_hetero_data(G)

Found 16 node types.
Add y label and train, val, test mask to data['Patient].
Found 996 edge types.
Add (patient,rel,protein) edge_weights to HeteroData
Conversion complete!


In [4]:
data

HeteroData(
  Gene={
    num_nodes=137,
    relevance=[137],
  },
  Abundance={
    num_nodes=408,
    relevance=[408],
  },
  BiologicalProcess={
    num_nodes=446,
    relevance=[446],
  },
  Activity={
    num_nodes=546,
    relevance=[546],
  },
  Pathology={
    num_nodes=126,
    relevance=[126],
  },
  MicroRna={
    num_nodes=46,
    relevance=[46],
  },
  Protein={
    num_nodes=1301,
    relevance=[1301],
  },
  Rna={
    num_nodes=111,
    relevance=[111],
  },
  Translocation={
    num_nodes=64,
    relevance=[64],
  },
  Reaction={
    num_nodes=13,
    relevance=[13],
  },
  Degradation={
    num_nodes=56,
    relevance=[56],
  },
  CellSecretion={
    num_nodes=30,
    relevance=[30],
  },
  CellSurfaceExpression={
    num_nodes=3,
    relevance=[3],
  },
  Complex={
    num_nodes=373,
    relevance=[373],
  },
  Composite={
    num_nodes=72,
    relevance=[72],
  },
  Patient={
    num_nodes=744,
    y=[744],
    train_mask=[744],
    val_mask=[744],
    test_mask=[744]

## Model Architecture

+ `nn.init.xavier_uniform_()`: 
  + initialize the tensor in-place using Xavier uniform initialization;
  + keep the variance of activations roughly the same across layers, 
  + helps with stable training and avoids vanishing/exploiding gradients.

In [ ]:
class GatedGNNLayer(MessagePassing):
    def __init__(self, in_channels, out_channels, heads=2):
        super().__init__(aggr='add', node_dim=0)
        self.heads = heads
        self.out_channels = out_channels

        if isinstance(in_channels, tuple):
            in_channels_src = in_channels[0]
            in_channels_dst = in_channels[1]
        else:
            in_channels_src = in_channels_dst = in_channels
        
        # project node input features to attention head space -> [num_edges, heads, out_channels]
        self.lin_l = nn.Linear(in_channels_dst, heads * out_channels)
        self.lin_r = nn.Linear(in_channels_src, heads * out_channels)
        
        # Relevance Controller: Learns Coeff_i [0, 1] for each node, 
        # act as a gate to control how much relevance_score is passed to dst
        self.relevance_controller = nn.Sequential(
            nn.Linear(in_channels_dst, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )
        # project rule
        #self.relevance_proj = nn.Linear(1, 1)
        self._coefficients = None
        
        # attention coefficient vector -> [1, heads, out_channels]
        # e_ij​ = LeakyReLU(a^T[ Wh_i​∣∣ Wh_j ​]), a is attention coefficient, weighting edges
        self.att_neural = nn.Parameter(torch.Tensor(1, heads, 2 * out_channels))
        nn.init.xavier_uniform_(self.att_neural)

    def forward(self, x, edge_index, relevance, edge_weight=None):
        # x should be a tuple (x_src, x_target) from HeteroConv
        if isinstance(x, tuple):
            # This is where your error "too many values to unpack" happens.
            #print(len(x)) # == 2
            x_src, x_target = x[0], x[1] 
        else:
            x_src = x_target = x
        
        # update target node coefficients
        coeffs = self.relevance_controller(x_target) 
        self._coefficients = coeffs

        # project node input features to attention head space -> [num_edges, heads, out_channels]
        x_l = self.lin_l(x_target).view(-1, self.heads, self.out_channels)
        x_r = self.lin_r(x_src).view(-1, self.heads, self.out_channels)
        
        # neural message passing for features
        out = self.propagate(edge_index, 
                             x=(x_r, x_l), 
                             rel=relevance, 
                             edge_weight=edge_weight, 
                             coeffs=coeffs)
        # return coeffs here to use in later loss calculation
        return out

    def message(self, x_j, x_i, rel_j, edge_weight, coeffs_i, index, ptr, size_i):
        
        # attention_neural score -> [num_edges, heads]
        alpha_neural = (torch.cat([x_i, x_j], dim=-1) * self.att_neural).sum(dim=-1)
        # deal with empty edge weight
        ew = edge_weight if edge_weight is not None else 1.0
        
        # rule is decided by node_relevance * expression (edge_weight) 
        #  and scaled by patient gate (coeff)
        alpha_rule = (rel_j * ew) * coeffs_i.view(-1, 1)
        
        # combine the attention terms to get a final attention coeeficient
        alpha = softmax(alpha_neural + alpha_rule, index, ptr, size_i)

        # dst node (protein -> patient) embeddings is scaled by attention coeeficent
        return x_j * alpha.unsqueeze(-1)
    
class GatedHeteroGNN(nn.Module):
    
    def __init__(self, data, in_channels, hidden_channels, out_channels, heads, num_layers=3, num_class=2):
        super().__init__()
        self.node_types = data.node_types
        
        # Learnable Embeddings for all nodes
        self.emb = nn.ModuleDict({node_type: torch.nn.Embedding(num_nodes, in_channels)
            for node_type, num_nodes in {nt: data[nt].num_nodes for nt in data.node_types}.items()
        })
        
        self.convs = nn.ModuleList()
        self.convs.append(HeteroConv({
            edge_type: GatedGNNLayer(in_channels, hidden_channels, heads)
                    for edge_type in data.edge_types
        }))

        # Stacked Parallel Layers
        if num_layers > 1:
            for _ in range(num_layers-1):
                self.convs.append(
                    HeteroConv({
                        edge_type: GatedGNNLayer(hidden_channels, hidden_channels, heads)
                        for edge_type in data.edge_types
                    })# 3 Layers for deep neighborhood relevance
                )
        '''
        self.convs.append(
            HeteroConv({
                        edge_type: GatedGNNLayer(hidden_channels, out_channels, heads)
                        for edge_type in data.edge_types
                    })
        )
        '''
        self.classifier = nn.Linear(out_channels, num_class)
        self.relevance_proj = nn.Linear(1,1)
        
    def forward(self, edge_index_dict, relevance_dict, weight_dict):
        x_dict = {nt: emb.weight.float() for nt, emb in self.emb.items()}
        
        all_coeffs = [] # To store for divergence loss

        for conv in self.convs:
            # 1. feature propagation
            # HeteroConv returns a dict of (h, c) tuples
            x_dict = conv(
                                x_dict, 
                                edge_index_dict, 
                                relevance_dict=relevance_dict, 
                                edge_weight_dict=weight_dict
                            )
            
            # unpack results
            print("\noutput of HeteroConv layer:")
            print(x_dict)
            #x_dict = {k: v[0] for k, v in results.items()}
            layer_coeffs = {}
            for edge_type, layer in conv.convs.items():
                dst_node_type = edge_type[2]
                # We save the coeffs associated with the destination of the edge
                layer_coeffs[dst_node_type] = layer._coefficients
            all_coeffs.append(layer_coeffs)
            
            x_dict = {k: F.elu(v) for k, v in x_dict.items()}
            
            # 2. SYMBOLIC PROPAGATION: Update relevance for the next layer
            relevance_dict = self.update_relevance(relevance_dict, edge_index_dict)
            
            # move the self.relevance_proj to this class
            relevance_dict = {k: torch.sigmoid(self.relevance_proj(v)) for k, v in relevance_dict.items()}
            

        out = self.classifier(x_dict['Patient'])
        
        return out, x_dict, all_coeffs
    
    def update_relevance(self, relevance_dict, edge_index_dict):
        """
        Aggregate relevance from neighboring nodes.
        Treat all paths equally, can aggregate in a weighted edge_type way.
        """
        new_rel_dict = {k: [] for k in relevance_dict.keys()}
        
        for edge_type, edge_index in edge_index_dict.items():
            src, _, dst = edge_type
            # Propagate relevance from source to destination
            rel_j = relevance_dict[src]
            res = scatter(rel_j[edge_index[0]], edge_index[1], dim=0, 
                        dim_size=relevance_dict[dst].size(0), reduce='mean')
            new_rel_dict[dst].append(res)
        
        # Average the relevance if a node receives it from multiple edge types
        for k in relevance_dict.keys():
            if len(new_rel_dict[k]) > 0:
                relevance_dict[k] = torch.stack(new_rel_dict[k]).mean(dim=0)
        return relevance_dict

In [40]:
class GatedHeteroGNN(nn.Module):
    
    def __init__(self, data, in_channels, hidden_channels, out_channels, heads, num_layers=3, num_class=2):
        super().__init__()
        self.node_types = data.node_types
        
        # Learnable Embeddings for all nodes
        self.emb = nn.ModuleDict({node_type: torch.nn.Embedding(num_nodes, in_channels)
            for node_type, num_nodes in {nt: data[nt].num_nodes for nt in data.node_types}.items()
        })
        
        self.convs = nn.ModuleList()
        self.convs.append(HeteroConv({
            edge_type: GatedGNNLayer(in_channels, hidden_channels, heads)
                    for edge_type in data.edge_types
        }))

        # Stacked Parallel Layers
        if num_layers > 1:
            for _ in range(num_layers-1):
                self.convs.append(
                    HeteroConv({
                        edge_type: GatedGNNLayer(hidden_channels, hidden_channels, heads)
                        for edge_type in data.edge_types
                    })# 3 Layers for deep neighborhood relevance
                )
        '''
        self.convs.append(
            HeteroConv({
                        edge_type: GatedGNNLayer(hidden_channels, out_channels, heads)
                        for edge_type in data.edge_types
                    })
        )
        '''
        self.classifier = nn.Linear(out_channels, num_class)
        self.relevance_proj = nn.Linear(1,1)
        
    def forward(self, edge_index_dict, relevance_dict, weight_dict):
        x_dict = {nt: emb.weight.float() for nt, emb in self.emb.items()}
        
        all_coeffs = [] # To store for divergence loss

        for conv in self.convs:
            # 1. feature propagation
            # HeteroConv returns a dict of (h, c) tuples
            x_dict = conv(
                                x_dict, 
                                edge_index_dict, 
                                relevance_dict=relevance_dict, 
                                edge_weight_dict=weight_dict
                            )
            
            # unpack results
            print("\noutput of HeteroConv layer:")
            print(x_dict)
            #x_dict = {k: v[0] for k, v in results.items()}
            layer_coeffs = {}
            for edge_type, layer in conv.convs.items():
                dst_node_type = edge_type[2]
                # We save the coeffs associated with the destination of the edge
                layer_coeffs[dst_node_type] = layer._coefficients
            all_coeffs.append(layer_coeffs)
            
            x_dict = {k: F.elu(v) for k, v in x_dict.items()}
            
            # 2. SYMBOLIC PROPAGATION: Update relevance for the next layer
            relevance_dict = self.update_relevance(relevance_dict, edge_index_dict)
            
            # move the self.relevance_proj to this class
            relevance_dict = {k: torch.sigmoid(self.relevance_proj(v)) for k, v in relevance_dict.items()}
            

        out = self.classifier(x_dict['Patient'])
        
        return out, x_dict, all_coeffs
    
    def update_relevance(self, relevance_dict, edge_index_dict):
        """
        Aggregate relevance from neighboring nodes.
        Treat all paths equally, can aggregate in a weighted edge_type way.
        """
        new_rel_dict = {k: [] for k in relevance_dict.keys()}
        
        for edge_type, edge_index in edge_index_dict.items():
            src, _, dst = edge_type
            # Propagate relevance from source to destination
            rel_j = relevance_dict[src]
            res = scatter(rel_j[edge_index[0]], edge_index[1], dim=0, 
                        dim_size=relevance_dict[dst].size(0), reduce='mean')
            new_rel_dict[dst].append(res)
        
        # Average the relevance if a node receives it from multiple edge types
        for k in relevance_dict.keys():
            if len(new_rel_dict[k]) > 0:
                relevance_dict[k] = torch.stack(new_rel_dict[k]).mean(dim=0)
        return relevance_dict

In [41]:
model = GatedHeteroGNN(
    data=data,
    in_channels=32,
    hidden_channels=128,
    out_channels=2,
    heads=2,
    num_layers=3
)
model

GatedHeteroGNN(
  (emb): ModuleDict(
    (Gene): Embedding(137, 32)
    (Abundance): Embedding(408, 32)
    (BiologicalProcess): Embedding(446, 32)
    (Activity): Embedding(546, 32)
    (Pathology): Embedding(126, 32)
    (MicroRna): Embedding(46, 32)
    (Protein): Embedding(1301, 32)
    (Rna): Embedding(111, 32)
    (Translocation): Embedding(64, 32)
    (Reaction): Embedding(13, 32)
    (Degradation): Embedding(56, 32)
    (CellSecretion): Embedding(30, 32)
    (CellSurfaceExpression): Embedding(3, 32)
    (Complex): Embedding(373, 32)
    (Composite): Embedding(72, 32)
    (Patient): Embedding(744, 32)
  )
  (convs): ModuleList(
    (0-2): 3 x HeteroConv(num_relations=996)
  )
  (classifier): Linear(in_features=2, out_features=2, bias=True)
  (relevance_proj): Linear(in_features=1, out_features=1, bias=True)
)

## Training

In [15]:
def train_step(model, data, edge_index_dict, relevance_dict, weight_dict, optimizer, lambdas):
    model.train()
    optimizer.zero_grad()
    
    # 1. Forward Pass
    result = model(edge_index_dict, relevance_dict, weight_dict)
    print(result)
    logits, h_dict, c_dict = result
    
    # --- Loss 1: Classification ---
    loss_cls = F.cross_entropy(logits[data['Patient'].train_mask], 
                               data['Patient'].y[data['Patient'].train_mask])
    
    # --- Loss 2: Link Prediction (DistMult style) ---
    loss_lp = 0
    for edge_type, edge_index in data.edge_index_dict.items():
        # Simple LP: dot product of source and target embeddings
        src, dst = edge_type[0], edge_type[2]
        pos_scores = (h_dict[src][edge_index[0]] * h_dict[dst][edge_index[1]]).sum(dim=-1)
        # Sample negatives
        neg_dst_idx = torch.randint(0, h_dict[dst].size(0), (edge_index.size(1),))
        neg_scores = (h_dict[src][edge_index[0]] * h_dict[dst][neg_dst_idx]).sum(dim=-1)
        loss_lp += F.margin_ranking_loss(pos_scores, neg_scores, torch.ones_like(pos_scores))
    
    # --- Loss 3: Divergence (Bimodal Regularization) ---
    # Force coefficients to be near 0 or 1
    patient_coeffs = c_dict['Patient']
    loss_div = torch.mean(patient_coeffs * (1 - patient_coeffs))
    
    # 4. Total Loss
    total_loss = (lambdas['cls'] * loss_cls + 
                  lambdas['lp'] * loss_lp + 
                  lambdas['div'] * loss_div)
    
    total_loss.backward()
    optimizer.step()
    
    return total_loss.item(), loss_cls.item(), loss_lp.item(), loss_div.item()

In [16]:
def contrastive_lp_loss(model, h_dict, edge_index_dict, margin=1.0):
    total_loss = 0
    for edge_type, edge_index in edge_index_dict.items():
        src_type, _, dst_type = edge_type
        
        # Anchor: Source nodes (e.g., Patients)
        # Positive: Real connected proteins
        # Negative: Randomly sampled proteins
        h_src = h_dict[src_type][edge_index[0]]
        h_pos = h_dict[dst_type][edge_index[1]]
        
        # Sample Negative
        neg_idx = torch.randint(0, h_dict[dst_type].size(0), (edge_index.size(1),), device=h_src.device)
        h_neg = h_dict[dst_type][neg_idx]
        
        # Triplet Loss: dist(anchor, pos) should be smaller than dist(anchor, neg)
        pos_dist = F.pairwise_distance(h_src, h_pos)
        neg_dist = F.pairwise_distance(h_src, h_neg)
        
        loss = F.relu(pos_dist - neg_dist + margin).mean()
        total_loss += loss
        
    return total_loss / len(edge_index_dict)

In [42]:
def initiate_relevance_dict(data, device):
    # prepare relevance_dict for training
    relevance_dict = {}
    for nt in data.node_types:
        if nt == 'Patient':
            relevance_dict[nt] = torch.full((data[nt].num_nodes, 1), 0).to(device)
        else:
            relevance_dict[nt] = data[nt].relevance.view(-1,1).to(device)
    relevance_dict = {k: v.float() for k, v in relevance_dict.items()}
    return relevance_dict

# edge_weight_dict
def initiate_edge_weight_dict(data, device):
    weight_dict = {}
    for edge_type in data.edge_types:
        num_edges = data[edge_type].edge_index.size(1)
        if 'Patient' in edge_type:
            weight_dict[edge_type] = data[edge_type].edge_weight.view(-1,1).to(device)
            #print(edge_type)
            #print(data[edge_type].edge_weight)
            #break
        else:
            # initiate other edge_weight as 1 to ensure neutral message flow
            weight_dict[edge_type] = torch.ones((num_edges,1)).to(device)
    weight_dict = {k: v.float() for k, v in weight_dict.items()}
    return weight_dict


In [44]:
from tqdm import tqdm


device = get_device()
edge_index_dict = {edge_type: data[edge_type].edge_index for edge_type in data.edge_types}
relevance_dict = initiate_relevance_dict(data, device)
weight_dict = initiate_edge_weight_dict(data, device)

optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.005,
        weight_decay=5e-4
    )
lambdas = {'cls':0.1,'lp':1, 'div': 0.005 }

loss_history = []
for epoch in tqdm(range(10), desc="Training GatedGNN"):
    losses = train_step(
        model=model,
        data=data,
        edge_index_dict=edge_index_dict,
        relevance_dict=relevance_dict,
        weight_dict=weight_dict,
        optimizer=optimizer,
        lambdas=lambdas
    )
    total_loss, cls_loss, lp_loss, div_loss = losses
    loss_history.append(losses)




Training GatedGNN:   0%|          | 0/10 [00:09<?, ?it/s]


output of HeteroConv layer:
{'Protein': tensor([[[ 0.3969, -0.1287, -0.2874,  ...,  1.1487, -0.3597, -0.6937],
         [-0.0509, -2.4082, -1.0389,  ...,  0.7885,  2.4346,  0.3041]],

        [[-0.1413,  0.3282, -0.0414,  ...,  0.9842, -1.8328,  0.0191],
         [-0.4035, -1.1385,  0.3533,  ...,  0.0399,  0.7671, -0.7593]],

        [[ 0.6061,  0.7582, -0.7347,  ...,  0.6761, -1.3644,  1.0254],
         [ 0.0305, -3.5235,  0.2460,  ...,  0.9562, -0.5395, -0.7242]],

        ...,

        [[-0.4965, -0.3282, -0.4113,  ...,  0.3516, -0.2541, -0.7204],
         [ 0.2541,  0.3085, -0.2295,  ..., -0.1077,  0.8531,  1.0096]],

        [[-0.4965, -0.3282, -0.4113,  ...,  0.3516, -0.2541, -0.7204],
         [ 0.2541,  0.3085, -0.2295,  ..., -0.1077,  0.8531,  1.0096]],

        [[ 1.2628,  0.1859, -0.6227,  ...,  0.6478, -2.3059,  0.2360],
         [ 0.1722, -0.0472,  0.2330,  ...,  0.1861,  0.1970, -0.1861]]],
       grad_fn=<SumBackward1>), 'Gene': tensor([[[-1.2153, -1.5116, -1.4051,  ...

ValueError: Encountered tensor with size 1301 in dimension 0, but expected size 2602.